In [8]:
import torch
from torch import nn, optim
from deep_river import classification
from river import metrics, ensemble, stream, preprocessing, compose
from river import base, tree, drift, utils, stats
import pandas as pd
#import numpy as np
import time
import os
import psutil

In [2]:
# Define o dispositivo globalmente
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo detectado: {device}")
if device == "cuda":
    print(f"GPU em uso: {torch.cuda.get_device_name(0)}")

Dispositivo detectado: cuda
GPU em uso: Quadro P620


In [3]:
class FlexibleNN(nn.Module):
    def __init__(self, n_features, n_classes, hidden_layers=[10]):
        super(FlexibleNN, self).__init__()
        layers = []
        in_dim = n_features
        
        for h_dim in hidden_layers:
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.ReLU())
            in_dim = h_dim
            
        layers.append(nn.Linear(in_dim, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [4]:
# Configurações de Heterogeneidade Parametrizadas
configs = [
    {"lr": 0.1,   "layers": [32],      "opt": optim.SGD},   # Reativo (Fast Learner)
    {"lr": 0.001, "layers": [64, 32],  "opt": optim.Adam},  # Estável (Long-term Memory)
    {"lr": 0.05,  "layers": [32, 16],  "opt": optim.SGD},   # Intermediário
    {"lr": 0.01,  "layers": [16],      "opt": optim.Adam}   # Conservador
]

ensemble_list = []
for i in range(50):
    cfg = configs[i % len(configs)]
    
    # Instancia o classificador base com o otimizador da config
    m = classification.Classifier(
        module=FlexibleNN(n_features=8, n_classes=2, hidden_layers=cfg["layers"]),
        loss_fn=nn.CrossEntropyLoss(),
        optimizer_fn=cfg["opt"], # Parametrizado aqui
        lr=cfg["lr"],
        is_feature_incremental=True,
        is_class_incremental=True,
        device=device
    )
    ensemble_list.append(m)

In [5]:
import collections
import statistics
import numpy as np
import torch
from river import base, stats, utils

class ARTEHeterogeneo(base.Ensemble, base.Classifier):
    """Adaptive Random Tree Ensemble (ARTE) adaptado para Heterogeneidade.
    
    Permite que cada membro do ensemble seja uma rede neural com arquitetura única.
    """
    def __init__(
        self,
        models: list, # Recebe a lista de modelos deep-river já instanciados
        drift_detector: base.DriftDetector = None,
        window_size: int = 500,
        n_rejections: int = 5,
        lambda_val: float = 6.0,
        seed: int = 1
    ):
        super().__init__(models=models)
        self.n_models = len(models)
        self.drift_detector = drift_detector
        self.window_size = window_size
        self.n_rejections = n_rejections
        self.lambda_val = lambda_val
        self._rng = np.random.RandomState(seed)
        self._avg_window_acc = 0.0
        
        # Inicializa os membros mantendo a referência ao modelo original para clonagem
        self._ensemble_members = []
        for m_base in models:
            self._ensemble_members.append({
                'model': m_base,
                'detector': drift_detector.clone(),
                'untrained_counts': collections.defaultdict(int),
                'window_acc': utils.Rolling(stats.Mean(), window_size=self.window_size),
                'drift_count': 0
            })

    def learn_one(self, x, y):
        all_accs = []
        for m in self._ensemble_members:
            y_pred = m['model'].predict_one(x)
            correct = (y == y_pred)
            
            # Lógica de Regularização Adaptativa do ARTE
            will_train = not correct
            if correct:
                m['untrained_counts'][y] += 1
                if self.n_rejections > 0 and m['untrained_counts'][y] >= self.n_rejections:
                    m['untrained_counts'][y] = 0
                    will_train = True
            
            if will_train:
                k = self._rng.poisson(self.lambda_val)
                for _ in range(k):
                    m['model'].learn_one(x, y)
            
            # Detecção de Drift Individual
            m['detector'].update(0 if correct else 1)
            if m['detector'].drift_detected:
                m['drift_count'] += 1
                # Reinicia preservando a arquitetura original (clone do deep-river)
                m['model'] = m['model'].clone() 
                m['detector'] = self.drift_detector.clone()
                m['window_acc'] = utils.Rolling(stats.Mean(), window_size=self.window_size)
            
            m['window_acc'].update(1 if correct else 0)
            all_accs.append(m['window_acc'].get())

        if all_accs:
            self._avg_window_acc = statistics.mean(all_accs)
        return self

    def predict_proba_one(self, x):
        combined_votes = collections.Counter()
        # Filtro de votação: apenas modelos acima da média de acurácia da janela
        eligible = [m for m in self._ensemble_members if m['window_acc'].get() >= self._avg_window_acc]
        if not eligible: eligible = self._ensemble_members

        for m in eligible:
            votes = m['model'].predict_proba_one(x)
            if votes:
                total = sum(votes.values())
                for cls, prob in votes.items():
                    combined_votes[cls] += prob / (total * len(eligible))
        return combined_votes

In [6]:
# Instancia o ARTE com a lista heterogênea
model_arte = ARTEHeterogeneo(
    models=ensemble_list,
    drift_detector=drift.ADWIN(delta=1e-3),
    window_size=500,
    seed=42
)

In [9]:
def get_memory_usage():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)

# 2. Configurações de Hardware e Dados
device = "cuda" if torch.cuda.is_available() else "cpu"
file_path = os.path.expanduser("~/moa/aldopaim/AdaptiveRegularizedEnsemble/datasets/elecNormNew.arff")
target_column = "class"
batch_size = 32  # Tamanho do mini-batch sugerido para GPU

oh_encoder = preprocessing.OneHotEncoder()
scaler = preprocessing.StandardScaler()

# # Nota: O StandardScaler também suporta learn_many nativamente no River
# pipeline = compose.Pipeline(
#     preprocessing.OneHotEncoder(sparse=False), # Adicionado sparse=False para compatibilizar o OHE com o Scaler
#     preprocessing.StandardScaler(),
#     model
# )

# Montamos o pipeline de pré-processamento
# # Nota: Não passamos o 'model' no pipeline compose para termos controle manual do cast
# preprocessor = compose.Pipeline(
#     preprocessing.OneHotEncoder(), 
#     preprocessing.StandardScaler()
# )

# 5. Loop de Treinamento em Mini-batch
label_map = {}
metric = metrics.Accuracy()
X_batch, y_batch = [], []
latencies = []

print(f"Iniciando Mini-batch (Size: {batch_size}) na GPU ({device})")
print(f"{'Instâncias':<12} | {'Acurácia':<10} | {'Lat. Lote (ms)':<15} | {'Lat. Inst (ms)':<15} | {'Drifts':<15} | {'Memória (MB)':<12}")
print("-" * 65)

dataset_stream = stream.iter_arff(file_path, target=target_column)
total_drifts = 0

for count, (x, y) in enumerate(dataset_stream, 1):
    # Mapeamento de classe
    if y not in label_map: label_map[y] = len(label_map)

    X_batch.append(x)
    y_batch.append(label_map[y])
    
    # Processa quando atingir o tamanho do lote
    if len(X_batch) == batch_size:
        batch_start = time.perf_counter()
        
        # Converte para DataFrame/Series (necessário para learn_many no River)
        X_df = pd.DataFrame(X_batch)
        y_ser = pd.Series(y_batch)

        # --- PASSO 1: One-Hot Encoding ---
        oh_encoder.learn_many(X_df)
        X_oh = oh_encoder.transform_many(X_df)

        # --- PASSO 2: Conversão Compulsória para Densa (FLOAT) ---
        # Isso resolve o TypeError impedindo que tipos Sparse cheguem ao Scaler
        X_oh = X_oh.astype(np.float64)
        if hasattr(X_oh, "sparse"): # Garante remoção de estrutura esparsa se houver
            X_oh = X_oh.sparse.to_dense()

        # --- PASSO 3: Escalonamento ---
        scaler.learn_many(X_oh)
        X_scaled = scaler.transform_many(X_oh)

        # Converte de volta para dicionários para compatibilidade com Ensemble
        # O custo aqui é baixo comparado ao ganho do OneHot em lote
        X_dicts = X_scaled.to_dict(orient='records')
        
        # --- PASSO 2: Predição e Treino (Incremental no Ensemble) ---
        for i in range(batch_size):
            # Test-then-Train
            # Cada predict_one/learn_one chamará o Deep-River na GPU
            y_pred = model_arte.predict_one(X_dicts[i])
            if y_pred is not None:
                metric.update(y_batch[i], y_pred)
            
            model_arte.learn_one(X_dicts[i], y_batch[i])

            # Verifica se algum detector ADWIN dentro do ensemble detectou mudança
            # # O ADWINBagging armazena os detectores em ._drift_detectors
            # for detector in model_arte._drift_detectors:
            #     if detector.drift_detected:
            #         total_drifts += 1
            #         print(f"![DRIFT] Detectado na instância {count - batch_size + i}")

        # Acessamos a lista interna de dicionários do ARTE
        total_drifts = sum(m['drift_count'] for m in model_arte._ensemble_members)
        
        # # 1. Predição em lote (Test-then-Train)
        # y_preds = pipeline.predict_many(X_df)
        # for yt, yp in zip(y_batch, y_preds):
        #     if yp is not None:
        #         metric.update(yt, yp)
        
        # 2. Treinamento em lote
        # pipeline.learn_many(X_df, y_ser)
        # model.learn_many(X_scaled, y_ser)
        
        # Medição de performance do lote
        batch_latency = (time.perf_counter() - batch_start) * 1000
        latencies.append(batch_latency)
        
        # Limpa o lote atual
        X_batch, y_batch = [], []
        
        # Reporte a cada 2000 instâncias (~62 lotes)
        if count % 2000 < batch_size:
            avg_lat_batch = sum(latencies[-20:]) / 20
            avg_lat_inst = avg_lat_batch / batch_size # Latência por instância
            mem = get_memory_usage()
            print(f"{count:<12} | {metric.get():>9.2%} | Unit: {avg_lat_inst:>6.2f}ms | Drifts: {total_drifts:<4} | Mem: {mem:>7.1f}MB")
            
        

print("-" * 65)
print(f"FINAL: Acc: {metric.get():.2%} | Classes mapeadas: {label_map}")

Iniciando Mini-batch (Size: 32) na GPU (cuda)
Instâncias   | Acurácia   | Lat. Lote (ms)  | Lat. Inst (ms)  | Drifts          | Memória (MB)
-----------------------------------------------------------------
2016         |    83.38% | Unit: 286.11ms | Drifts: 0    | Mem:   913.6MB
4000         |    82.50% | Unit: 445.94ms | Drifts: 17   | Mem:   941.5MB
6016         |    81.83% | Unit: 570.78ms | Drifts: 25   | Mem:   950.6MB


KeyboardInterrupt: 